# ECE 382V extended flakiness run (Colab)

Reruns every test the static scan flagged as risky (about 330 tests across 36 suites) **1,000 times each**, 100 times in each of 10 time zones, on Colab's Linux VM.

**Before you start**
1. Put `ece382v_colab_bundle.tar.gz` in Google Drive at `MyDrive/ece382v_flaky/`.
2. Runtime > Change runtime type > **CPU** is enough (no GPU needed).
3. Run all cells. Expect about 4 to 6 hours on the free 2 vCPU runtime.

**Keeping it alive.** Free Colab stops after at most 12 hours, disconnects after about 90 minutes without browser activity, and does not keep running if you close the tab. Keep this tab open and the laptop awake (plugged in, lid open). If it disconnects anyway, just reconnect and **Run all** again. Every finished chunk is saved to Drive, so it continues where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ece382v_flaky'
BUNDLE = f'{DRIVE_DIR}/ece382v_colab_bundle.tar.gz'
OUT = f'{DRIVE_DIR}/results_colab'      # every finished chunk lands here, so restarts resume
COUNT = 100                             # repeats per test per time zone, 10 zones = 1,000 runs per test
WORKERS = 2                             # one per vCPU keeps the load comparable to the laptop runs
import os; assert os.path.exists(BUNDLE), 'Upload ece382v_colab_bundle.tar.gz to MyDrive/ece382v_flaky first'
!nproc; python3 --version

In [ ]:
# Setup (about 3 minutes the first time, skipped on a rerun) and the full run.
# Output streams below. Safe to rerun this cell after a disconnect.
!tar xzf "$BUNDLE" -C /content ece382v-flaky-llm-tests/colab/setup_and_run.sh
!bash /content/ece382v-flaky-llm-tests/colab/setup_and_run.sh "$BUNDLE" /content "$OUT" $COUNT $WORKERS 2>&1 | grep --line-buffered -vE '^\s*$'

In [ ]:
# Summary of everything saved so far (works even if the run is still partial).
!cd /content/ece382v-flaky-llm-tests && python3 scripts/extended.py --summarize --out "$OUT"
import json; s = json.load(open(f'{OUT}/summary.json'))
print(f"{s['chunks']} chunks, {s['executions']:,} test executions, {s['flaky']} flaky tests")

When it finishes, the results are already in Drive at `MyDrive/ece382v_flaky/results_colab/` (one JSON per chunk plus `summary.json`). Download that folder, or tell Claude it is done and it can read it from there.